# 02 — Data Modeling & Warehouse Load

Goal: stand up the star schema in Postgres and load the cleaned parquet snapshots into it.

Prereq: `.env` configured (copy `.env.example`) and Postgres running.  
Inputs: `data/processed/*.parquet`, `sql/*.sql`  
Outputs: populated `fact_*` and `dim_*` tables.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

sys.path.append('..')

from src import db  # noqa: E402
from src.config import PROJECT_ROOT  # noqa: E402

SQL_DIR = PROJECT_ROOT / 'sql'

## 1. Verify the database connection

In [ ]:
assert db.test_connection(), 'Cannot reach Postgres — check .env / service.'

## 2. Create the star schema (DDL)
Runs `sql/00_create_star_schema.sql` — idempotent (drops + recreates).

In [ ]:
db.run_script(str(SQL_DIR / '00_create_star_schema.sql'))
print('star schema created')

## 3. Load cleaned parquet snapshots into staging
Bulk-load each processed table into a matching raw schema for the ETL below.

In [ ]:
import pandas as pd

from src.data_loader import load_processed, OLIST_FILES

engine = db.get_engine()
for table_name in OLIST_FILES.values():
    try:
        df = load_processed(table_name)
    except FileNotFoundError:
        continue
    df.to_sql(f'stg_{table_name}', engine, if_exists='replace', index=False)
    print(f'loaded stg_{table_name}: {len(df):,} rows')

## 4. Populate dims & facts
TODO: transform staging tables into the star schema via `INSERT ... SELECT`.
For each dimension/fact, write the transformation SQL here, then run via `db.run_script`.

In [ ]:
# Example: dim_date from the orders purchase timestamps.
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text('TRUNCATE dim_date'))
    conn.execute(text('''
        INSERT INTO dim_date (date_key, day, month, quarter, year, day_of_week, is_weekend)
        SELECT DISTINCT
            d::date,
            EXTRACT(day FROM d)::int,
            EXTRACT(month FROM d)::int,
            EXTRACT(quarter FROM d)::int,
            EXTRACT(year FROM d)::int,
            EXTRACT(dow FROM d)::int,
            EXTRACT(dow FROM d) IN (0, 6)
        FROM (
            SELECT DISTINCT order_purchase_timestamp::date AS d
            FROM stg_orders
        ) s
    '''))
    print('dim_date loaded')

## 5. Build KPI views
Run `sql/02_kpi_views.sql` so the dashboard has ready-made marts.

In [ ]:
db.run_script(str(SQL_DIR / '02_kpi_views.sql'))
print('KPI materialised views created')